In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    RobustScaler,
    LabelEncoder,
    SplineTransformer,
    PolynomialFeatures,
    FunctionTransformer,
)
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, RidgeCV
from sklearn.ensemble import VotingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pygam import LogisticGAM, s

df_train = pd.read_csv("QSAR_challenge_train_80pct.csv")
df_test = pd.read_csv("QSAR_challenge_test_20pct_NO_TARGETS.csv")

continuous_cols = ["nHM", "piPC09", "PCD", "X2Av", "MLOGP", "ON1V"]
fragment_cols = ["N-072", "B02[C-N]", "F04[C-O]"]
features = continuous_cols + fragment_cols

X_train_class = df_train[features]
le = LabelEncoder()
y_class_train = le.fit_transform(df_train["classification_target"])

prep_standard = ColumnTransformer(
    [
        (
            "cont",
            Pipeline(
                [
                    (
                        "imp",
                        IterativeImputer(
                            estimator=BayesianRidge(), max_iter=15, random_state=42
                        ),
                    ),
                    ("scl", RobustScaler()),
                ]
            ),
            continuous_cols,
        ),
        (
            "frag",
            Pipeline(
                [
                    ("imp", SimpleImputer(strategy="constant", fill_value=0.0)),
                    ("scl", RobustScaler()),
                ]
            ),
            fragment_cols,
        ),
    ]
)

X_train_class_p = prep_standard.fit_transform(X_train_class)
classes = np.unique(y_class_train)

gam_class_models = {}
for c in classes:
    gam = LogisticGAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6) + s(7) + s(8))
    gam.gridsearch(
        X_train_class_p,
        (y_class_train == c).astype(float),
        lam=np.logspace(-3, 3, 20),
        progress=False,
    )
    gam_class_models[c] = gam

df_train_filtered = df_train.copy()
for col in continuous_cols:
    min_val = df_test[col].min()
    max_val = df_test[col].max()
    df_train_filtered = df_train_filtered[
        (df_train_filtered[col] >= min_val) & (df_train_filtered[col] <= max_val)
    ]

X_train_reg = df_train_filtered[features]
y_reg_train = df_train_filtered["regression_target"]

frag_pipe = Pipeline(
    [
        (
            "imp",
            IterativeImputer(
                estimator=BayesianRidge(), max_iter=15, min_value=0, random_state=42
            ),
        ),
        ("round", FunctionTransformer(np.round, validate=False)),
        ("scl", RobustScaler()),
    ]
)

prep_spline = ColumnTransformer(
    [
        (
            "cont_spline",
            Pipeline(
                [
                    (
                        "imp",
                        IterativeImputer(
                            estimator=BayesianRidge(), max_iter=15, random_state=42
                        ),
                    ),
                    (
                        "spline",
                        SplineTransformer(
                            n_knots=5,
                            degree=3,
                            extrapolation="linear",
                            include_bias=False,
                        ),
                    ),
                    ("scl", RobustScaler()),
                ]
            ),
            continuous_cols,
        ),
        ("frag", frag_pipe, fragment_cols),
    ]
)
pipe_spline = Pipeline(
    [("prep", prep_spline), ("model", RidgeCV(alphas=np.logspace(-3, 3, 50)))]
)

signed_log_tf = FunctionTransformer(
    lambda X: np.sign(X) * np.log1p(np.abs(X)), validate=False
)

prep_polylog = ColumnTransformer(
    [
        (
            "cont_polylog",
            Pipeline(
                [
                    (
                        "imp",
                        IterativeImputer(
                            estimator=BayesianRidge(), max_iter=15, random_state=42
                        ),
                    ),
                    ("log", signed_log_tf),
                    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                    ("scl", RobustScaler()),
                ]
            ),
            continuous_cols,
        ),
        ("frag", frag_pipe, fragment_cols),
    ]
)
pipe_polylog = Pipeline(
    [("prep", prep_polylog), ("model", RidgeCV(alphas=np.logspace(-3, 3, 50)))]
)

ensemble_reg = VotingRegressor(
    estimators=[("spline", pipe_spline), ("polylog", pipe_polylog)]
)

ensemble_reg.fit(X_train_reg, y_reg_train)

/Users/otsep/Desktop/Personal/KSE/ml/personal/KSE_ML_for_bioinformatics/.venv/lib/python3.11/site-packages/pygam/links.py:137: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
/Users/otsep/Desktop/Personal/KSE/ml/personal/KSE_ML_for_bioinformatics/.venv/lib/python3.11/site-packages/pygam/pygam.py:631: RuntimeWarning: overflow encountered in square
  self.link.gradient(mu, self.distribution) ** 2
/Users/otsep/Desktop/Personal/KSE/ml/personal/KSE_ML_for_bioinformatics/.venv/lib/python3.11/site-packages/pygam/pygam.py:631: RuntimeWarning: invalid value encountered in multiply
  self.link.gradient(mu, self.distribution) ** 2
/Users/otsep/Desktop/Personal/KSE/ml/personal/KSE_ML_for_bioinformatics/.venv/lib/python3.11/site-packages/pygam/links.py:121: RuntimeWarning: overflow encountered in exp
  elp = np.exp(lp)
/Users/otsep/Desktop/Personal/KSE/ml/personal/KSE_ML_for_bioinformatics/.venv/lib/python3.11/site-packages/pygam/links.py:122: R

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('spline', ...), ('polylog', ...)]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
estimators_ estimators_: list of regressorsThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'.,list,"[Pipeline(step...red_error'))]), Pipeline(step...m_state=42))])]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['nHM','piPC09','PCD',...,'N-072','B02[C-N]','F04[C-O]']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,9
named_estimators_ named_estimators_: :class:`~sklearn.utils.Bunch`Attribute to access any fitted sub-estimators by name... versionadded:: 0.20,Bunch,{'spline': Pi..._state=42))])}
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cont_spline', ...), ('frag', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'


In [30]:
df_test = pd.read_csv("QSAR_challenge_test_20pct_NO_TARGETS.csv")
X_new = df_test[features]

X_new_class_p = prep_standard.transform(X_new)
probs_new = np.zeros((X_new_class_p.shape[0], len(classes)))

for i, c in enumerate(classes):
    probs_new[:, i] = gam_class_models[c].predict_proba(X_new_class_p)
df_test["classification_prediction"] = le.inverse_transform(
    classes[np.argmax(probs_new, axis=1)]
)
df_test["regression_prediction"] = ensemble_reg.predict(X_new)

display(df_test.head())
df_test[["sample_id", "regression_prediction", "classification_prediction"]].to_csv(
    "df_new.csv"
)

,sample_id,nHM,piPC09,PCD,X2Av,MLOGP,ON1V,N-072,B02[C-N],F04[C-O],classification_prediction,regression_prediction
0,78-79-5,0.0,0.000,0.45,0.26,1.97,0.63,0.0,0.0,0.0,1,0.891563
1,16606-02-3,3.0,6.173,2.31,0.20,5.47,1.23,0.0,0.0,0.0,1,3.883802
2,111-44-4,2.0,0.000,0.00,0.35,1.59,0.81,0.0,0.0,0.0,1,0.651871
3,107-05-1,1.0,0.000,0.24,0.38,1.67,0.41,0.0,0.0,NaN,1,0.401693
4,555-03-3,0.0,0.000,1.48,0.14,1.70,0.88,0.0,1.0,5.0,1,0.767106
